In [155]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_validate, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import make_scorer, accuracy_score, f1_score

scoring_dict = {
    'accuracy': 'accuracy',
    'f1_macro': 'f1_macro',
    'f1_weighted': 'f1_weighted'
}



# Suport functions

In [156]:
def confusion(true, pred):
    """
    Function for pretty printing confusion matrices
    """
    true.name = 'target'
    pred.name = 'predicted'
    cm = pd.crosstab(true.reset_index(drop=True), pred.reset_index(drop=True))
    cm = cm[cm.index]
    return cm

# Data loading

In [157]:
ILDS = pd.read_csv("scaled_train_fs.csv", delimiter=',', header=None)


ILDS.columns = ['Age', 'TB', 'Alkphos', 'Sgot', 'ALB', 'AR', 'BilRatio', 'Female', 'Target']

display(ILDS)

,Age,TB,Alkphos,Sgot,ALB,AR,BilRatio,Female,Target
0,0.173844,1.109406,0.362037,0.487504,-0.903219,-1.399329,1.203258,0,0
1,-0.379308,0.226878,-0.573270,0.288250,1.482342,1.488320,0.927290,0,0
2,-1.362690,-0.430090,-0.232378,0.575302,-0.024328,0.212382,-0.353374,0,0
3,-0.194924,-0.795164,-0.925510,0.589292,0.101228,0.413846,-0.458710,1,0
4,0.542612,2.761282,1.783803,-0.293097,0.352339,-0.459164,1.153956,1,0
...,...,...,...,...,...,...,...,...,...
444,-0.440770,-0.658488,-1.023732,-0.828504,-0.024328,0.313114,-0.722050,1,1
445,1.095764,-0.537931,-0.417227,-0.072655,0.980118,3.066455,-0.926871,0,1
446,0.911380,-0.795164,-0.680130,-0.690433,0.477895,0.212382,-0.458710,0,1
447,-0.625154,-0.537931,-0.460631,-0.985900,0.603451,0.883929,-0.926871,0,1


In [158]:
X = ILDS.loc[:, ILDS.columns != 'Target']
y = ILDS['Target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify = y, random_state=1234)

In [159]:
from imblearn.under_sampling import RandomUnderSampler

rus = RandomUnderSampler(random_state=42)
X_train, y_train = rus.fit_resample(X_train, y_train)


In [160]:
results_df = pd.DataFrame(index=[], columns= ['Accuracy', 'F1 Macro', 'Precision Macro', 'Recall Macro'])

# Decision tree

# Random forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=1234, class_weight='balanced')
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

In [162]:
confusion(y_train, pd.Series(rf.predict(X_train)))

predicted,0,1
target,,
0,101,0
1,0,101


In [163]:
confusion(y_test, pd.Series(rf.predict(X_test)))     

predicted,0,1
target,,
0,45,20
1,7,18


In [164]:
cross_val_results = pd.DataFrame(cross_validate(rf , X_train, y_train, cv = 5, 
                            scoring = ['accuracy', 'f1_macro', 'precision_macro', 'recall_macro'] ))

results_df.loc['RF',:] = cross_val_results[['test_accuracy', 'test_f1_macro',
       'test_precision_macro', 'test_recall_macro']].mean().values
results_df

,Accuracy,F1 Macro,Precision Macro,Recall Macro
RF,0.727805,0.724911,0.739113,0.727619


# Gradient Boosting

In [165]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.utils.class_weight import compute_sample_weight

weights = compute_sample_weight(class_weight='balanced', y=y_train)
gb = GradientBoostingClassifier(n_estimators=100, random_state=42)
gb.fit(X_train, y_train, sample_weight=weights)
gb_pred = gb.predict(X_test)

In [166]:
cross_val_results = pd.DataFrame(cross_validate(gb , X_train, y_train, cv = 5, 
                            scoring = ['accuracy', 'f1_macro', 'precision_macro', 'recall_macro'] ))

results_df.loc['GB',:] = cross_val_results[['test_accuracy', 'test_f1_macro',
       'test_precision_macro', 'test_recall_macro']].mean().values
results_df

,Accuracy,F1 Macro,Precision Macro,Recall Macro
RF,0.727805,0.724911,0.739113,0.727619
GB,0.663415,0.661354,0.667965,0.662857


# Voting classifier

In [167]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import VotingClassifier

log_clf = LogisticRegression(class_weight="balanced")
svc_clf = SVC(probability=True, class_weight="balanced")
rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
voting = VotingClassifier(estimators=[
    ('lr', log_clf), ('svc', svc_clf), ('rf', rf)
], voting='soft')

voting.fit(X_train, y_train)
voting_pred = voting.predict(X_test)


In [168]:
confusion(y_train, pd.Series(voting.predict(X_train)))

predicted,0,1
target,,
0,86,15
1,7,94


In [169]:
confusion(y_test, pd.Series(voting.predict(X_test)))

predicted,0,1
target,,
0,44,21
1,7,18


In [170]:
cross_val_results = pd.DataFrame(cross_validate(voting , X_train, y_train, cv = 5, 
                            scoring = ['accuracy', 'f1_macro', 'precision_macro', 'recall_macro'] ))

results_df.loc['CLF',:] = cross_val_results[['test_accuracy', 'test_f1_macro',
       'test_precision_macro', 'test_recall_macro']].mean().values
results_df

,Accuracy,F1 Macro,Precision Macro,Recall Macro
RF,0.727805,0.724911,0.739113,0.727619
GB,0.663415,0.661354,0.667965,0.662857
CLF,0.708171,0.70446,0.7182,0.707619


# XGBoost

In [171]:
from xgboost import XGBClassifier

xgb = XGBClassifier(scale_pos_weight=1/3)
xgb.fit(X_train, y_train)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=None, num_parallel_tree=None, ...)

In [172]:
cross_val_results = pd.DataFrame(cross_validate(xgb , X_train, y_train, cv = 5, 
                            scoring = ['accuracy', 'f1_macro', 'precision_macro', 'recall_macro'] ))

results_df.loc['XGB',:] = cross_val_results[['test_accuracy', 'test_f1_macro',
       'test_precision_macro', 'test_recall_macro']].mean().values
results_df

,Accuracy,F1 Macro,Precision Macro,Recall Macro
RF,0.727805,0.724911,0.739113,0.727619
GB,0.663415,0.661354,0.667965,0.662857
CLF,0.708171,0.70446,0.7182,0.707619
XGB,0.663171,0.656881,0.675138,0.662381


In [173]:
confusion(y_train, pd.Series(xgb.predict(X_train)))

predicted,0,1
target,,
0,101,0
1,0,101


In [174]:
confusion(y_test, pd.Series(xgb.predict(X_test)))

predicted,0,1
target,,
0,49,16
1,11,14


# AdaBoosting

In [175]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier


In [176]:
base = DecisionTreeClassifier(max_depth=1, class_weight='balanced')
ada = AdaBoostClassifier(estimator=base, n_estimators=100, random_state=42)
ada.fit(X_train, y_train)


C:\Users\haoka\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\ensemble\_weight_boosting.py:519: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(


AdaBoostClassifier(estimator=DecisionTreeClassifier(class_weight='balanced',
                                                    max_depth=1),
                   n_estimators=100, random_state=42)

In [177]:
cross_val_results = pd.DataFrame(cross_validate(ada , X_train, y_train, cv = 5, 
                            scoring = ['accuracy', 'f1_macro', 'precision_macro', 'recall_macro'] ))

results_df.loc['ADA',:] = cross_val_results[['test_accuracy', 'test_f1_macro',
       'test_precision_macro', 'test_recall_macro']].mean().values
results_df

C:\Users\haoka\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\ensemble\_weight_boosting.py:519: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
C:\Users\haoka\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\ensemble\_weight_boosting.py:519: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
C:\Users\haoka\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\ensemble\_weight_boosting.py:519: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent th

,Accuracy,F1 Macro,Precision Macro,Recall Macro
RF,0.727805,0.724911,0.739113,0.727619
GB,0.663415,0.661354,0.667965,0.662857
CLF,0.708171,0.70446,0.7182,0.707619
XGB,0.663171,0.656881,0.675138,0.662381
ADA,0.653049,0.651103,0.659909,0.653571


In [178]:
confusion(y_train, pd.Series(ada.predict(X_train)))

predicted,0,1
target,,
0,97,4
1,1,100


In [179]:
confusion(y_test, pd.Series(ada.predict(X_test)))

predicted,0,1
target,,
0,43,22
1,8,17


# Extra Trees

In [180]:
rf_model = ExtraTreesClassifier(class_weight='balanced')

ntrees = [150, None]
max_depth = [100, None]
min_samples_split = [4, 6]
min_samples_leaf = [2, 4]
balance = [None, 'balanced', 'balanced_subsample']

trc = GridSearchCV(estimator=rf_model,
                   scoring=scoring_dict,
                   param_grid={
                       'n_estimators': ntrees,
                       'max_depth': max_depth,
                       'min_samples_split': min_samples_split,
                       'min_samples_leaf': min_samples_leaf,
                       'class_weight': balance
                   },
                   cv=5,
                   return_train_score=True,
                   refit=False,
                   n_jobs=-1)

model_5CV = trc.fit(X_train, y_train)


C:\Users\haoka\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\model_selection\_validation.py:547: FitFailedWarning: 
120 fits failed out of a total of 240.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
120 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\haoka\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\model_selection\_validation.py", line 895, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "C:\Users\haoka\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-p

In [181]:
# Convert results to DataFrame
cv_results_df = pd.DataFrame(trc.cv_results_)

# Display the keys to see all metrics stored
print(cv_results_df.columns)


# Find index of the best mean test f1_macro score
best_idx = cv_results_df['mean_test_f1_macro'].idxmax()

# Get the best row
best_result = cv_results_df.loc[best_idx]

# Print best score and parameters
print("Best F1_macro:", best_result['mean_test_f1_macro'])
print("Best params:", best_result['params'])


# Take mean metrics from best ExtraTrees configuration
# results_df.loc['ExtraTrees', :] = best_result[[
#     'mean_test_accuracy', 
#     'mean_test_f1_macro', 
#     'mean_test_precision_macro', 
#     'mean_test_recall_macro'
# ]].values

results_df

Index(['mean_fit_time', 'std_fit_time', 'mean_score_time', 'std_score_time',
       'param_class_weight', 'param_max_depth', 'param_min_samples_leaf',
       'param_min_samples_split', 'param_n_estimators', 'params',
       'split0_test_accuracy', 'split1_test_accuracy', 'split2_test_accuracy',
       'split3_test_accuracy', 'split4_test_accuracy', 'mean_test_accuracy',
       'std_test_accuracy', 'rank_test_accuracy', 'split0_train_accuracy',
       'split1_train_accuracy', 'split2_train_accuracy',
       'split3_train_accuracy', 'split4_train_accuracy', 'mean_train_accuracy',
       'std_train_accuracy', 'split0_test_f1_macro', 'split1_test_f1_macro',
       'split2_test_f1_macro', 'split3_test_f1_macro', 'split4_test_f1_macro',
       'mean_test_f1_macro', 'std_test_f1_macro', 'rank_test_f1_macro',
       'split0_train_f1_macro', 'split1_train_f1_macro',
       'split2_train_f1_macro', 'split3_train_f1_macro',
       'split4_train_f1_macro', 'mean_train_f1_macro', 'std_train_f1_macr

,Accuracy,F1 Macro,Precision Macro,Recall Macro
RF,0.727805,0.724911,0.739113,0.727619
GB,0.663415,0.661354,0.667965,0.662857
CLF,0.708171,0.70446,0.7182,0.707619
XGB,0.663171,0.656881,0.675138,0.662381
ADA,0.653049,0.651103,0.659909,0.653571


# Final test export

In [182]:
display(ILDS_test)

,Age,TB,Alkphos,Sgot,ALB,AR,BilRatio,Female
0,-2.100226,-0.795164,1.907027,-0.567456,1.356786,1.555475,-1.512071,0
1,1.034303,0.171538,-0.117670,1.320148,1.105674,-0.459164,1.121330,0
2,0.911380,-0.795164,-0.643898,-1.387576,1.356786,0.548155,-0.458710,0
3,0.911380,1.351361,-0.212816,3.236675,0.101228,-0.526319,1.056650,0
4,0.173844,-0.537931,-0.631959,0.132669,-0.526551,-0.123391,-0.926871,1
...,...,...,...,...,...,...,...,...
111,1.464532,-0.658488,-1.009414,0.424923,2.235677,1.555475,-0.722050,1
112,-0.625154,2.916471,-1.637484,2.899368,-1.405442,-2.138031,0.826950,0
113,-0.194924,-0.658488,-1.524844,-0.828504,-0.149884,-0.794938,-0.722050,0
114,-1.669996,-0.952944,1.707761,-0.388323,0.101228,-0.794938,-0.107590,0


In [183]:
rf = RandomForestClassifier(n_estimators=100, random_state=1234, class_weight='balanced')
rf.fit(X_train, y_train)

ILDS_test = pd.read_csv("scaled_test_fs.csv", delimiter=',', header=None)

ILDS_test.columns = ['Age', 'TB', 'Alkphos', 'Sgot', 'ALB', 'AR', 'BilRatio', 'Female']

X_test = ILDS_test.loc[:,:'Female']

ILDS_test['Label'] = rf.predict(X_test)


ILDS_test.index = ILDS_test.index + 1
ILDS_test.index.name = 'ID'

ILDS_test['Label'].to_csv('random_forest_fs.csv', index=True)